# Two Moons with NSF

Train a Neural Spline Flow on the two-moons toy dataset and visualize the learned density.

In [2]:
:dep flowrs = { path = "..", default-features = false, features = ["ndarray"] }
:dep burn = { version = "0.20", features = ["autodiff", "ndarray"] }
:dep plotpy = "1"
:dep rand = "0.8"
:dep rand_distr = "0.4"

## Generate Two Moons Dataset

In [3]:
use rand::rngs::StdRng;
use rand::{Rng, SeedableRng};
use rand_distr::{Distribution, Normal};

fn generate_two_moons(n: usize, noise: f64, seed: u64) -> (Vec<f64>, Vec<f64>) {
    let mut rng = StdRng::seed_from_u64(seed);
    let normal = Normal::new(0.0, noise).unwrap();
    let mut xs = Vec::with_capacity(n);
    let mut ys = Vec::with_capacity(n);
    let half = n / 2;
    for i in 0..half {
        let angle = std::f64::consts::PI * (i as f64) / (half as f64);
        xs.push(angle.cos() + normal.sample(&mut rng));
        ys.push(angle.sin() + normal.sample(&mut rng));
    }
    for i in 0..(n - half) {
        let angle = std::f64::consts::PI * (i as f64) / ((n - half) as f64);
        xs.push(1.0 - angle.cos() + normal.sample(&mut rng));
        ys.push(1.0 - angle.sin() - 0.5 + normal.sample(&mut rng));
    }
    (xs, ys)
}

let n_data: usize = 10_000;
let (data_x, data_y) = generate_two_moons(n_data, 0.05, 42u64);
println!("Generated {} points", n_data);

Generated 10000 points


## Plot Training Data

In [4]:
use plotpy::{Curve, Plot};

let mut scatter = Curve::new();
scatter
    .set_line_style("None")
    .set_line_alpha(0.3)
    .set_marker_style("o")
    .set_marker_size(1.0)
    .set_marker_color("#4285f4")
    .set_marker_line_color("#4285f4");
scatter.draw(&data_x, &data_y);

let mut plot = Plot::new();
plot.add(&scatter)
    .set_equal_axes(true)
    .set_labels("x", "y")
    .set_title("Two Moons Training Data");
plot.show_in_jupyter("training_data.svg")

Ok(())

## Train NSF, Sample, and Visualize

In [ ]:
use burn::prelude::*;
use burn::backend::{Autodiff, NdArray};
use burn::module::AutodiffModule;
use burn::optim::{AdamConfig, GradientsParams, Optimizer};
use rand::seq::SliceRandom;
use flowrs::NsfConfig;

type B = Autodiff<NdArray>;
type BI = NdArray;

let (sample_x, sample_y, densities, nx, ny, min_x, max_x, min_y, max_y):
    (Vec<f64>, Vec<f64>, Vec<f64>, usize, usize, f64, f64, f64, f64) = {

    let device = <BI as Backend>::Device::default();

    let points: Vec<[f32; 2]> = data_x.iter().zip(data_y.iter())
        .map(|(&x, &y)| [x as f32, y as f32])
        .collect();

    fn sample_batch(points: &[[f32; 2]], batch_size: usize, device: &<BI as Backend>::Device, rng: &mut StdRng) -> Tensor<B, 2> {
        let chosen: Vec<[f32; 2]> = points.choose_multiple(rng, batch_size).cloned().collect();
        let flat: Vec<f32> = chosen.iter().flat_map(|p| p.iter().copied()).collect();
        Tensor::from_floats(TensorData::new(flat, [batch_size, 2]), device)
    }

    // --- Train ---
    let config = NsfConfig::new(2, 6, vec![128, 128])
        .with_num_bins(8usize)
        .with_tail_bound(3.0f32);
    let mut model: flowrs::Nsf<B> = config.init::<B>(&device);
    let mut optim = AdamConfig::new().init();
    let mut rng = StdRng::seed_from_u64(42u64);

    let num_epochs: usize = 500;
    let batch_size: usize = 512;
    let batches_per_epoch: usize = 10;
    let lr: f64 = 5e-4;

    for epoch in 1..=num_epochs {
        let mut epoch_loss = 0.0_f32;
        for _ in 0..batches_per_epoch {
            let batch = sample_batch(&points, batch_size, &device, &mut rng);
            let log_prob = model.log_prob(batch);
            let loss = -log_prob.mean();
            let lv: f32 = loss.clone().into_scalar().elem();
            epoch_loss += lv;
            let grads = loss.backward();
            let grads = GradientsParams::from_grads(grads, &model);
            let progress = (epoch - 1) as f64 / num_epochs as f64;
            let elr = lr * 0.5 * (1.0 + (std::f64::consts::PI * progress).cos());
            model = optim.step(elr, model, grads);
        }
        if epoch % 100 == 0 || epoch == 1 {
            let avg = epoch_loss / batches_per_epoch as f32;
            println!("Epoch {:>4}/{}  NLL: {:.4}", epoch, num_epochs, avg);
        }
    }
    println!("Training complete.");

    // --- Sample ---
    let model_valid: flowrs::Nsf<BI> = model.valid();

    let n_samples: usize = 5000;
    let z = Tensor::<BI, 2>::random(
        [n_samples, 2],
        burn::tensor::Distribution::Normal(0.0, 1.0),
        &device,
    );
    let samples = model_valid.inverse(z);
    let svals: Vec<f32> = samples.to_data().to_vec().unwrap();
    let sample_x: Vec<f64> = (0..n_samples).map(|i| svals[i * 2] as f64).collect();
    let sample_y: Vec<f64> = (0..n_samples).map(|i| svals[i * 2 + 1] as f64).collect();

    // --- Density grid ---
    let nx: usize = 100;
    let ny: usize = 100;
    let (min_x, max_x, min_y, max_y) = (-1.5_f64, 2.5, -1.0, 1.5);

    let mut flat_points: Vec<f32> = Vec::with_capacity(nx * ny * 2);
    for iy in 0..ny {
        let yy = min_y + (max_y - min_y) * iy as f64 / (ny - 1) as f64;
        for ix in 0..nx {
            let xx = min_x + (max_x - min_x) * ix as f64 / (nx - 1) as f64;
            flat_points.push(xx as f32);
            flat_points.push(yy as f32);
        }
    }
    let n_grid = nx * ny;
    let chunk_size: usize = 1000;
    let mut densities: Vec<f64> = Vec::with_capacity(n_grid);
    for start in (0..n_grid).step_by(chunk_size) {
        let end = (start + chunk_size).min(n_grid);
        let blen = end - start;
        let flat: Vec<f32> = flat_points[start * 2..end * 2].to_vec();
        let tensor = Tensor::<BI, 2>::from_floats(
            TensorData::new(flat, [blen, 2]),
            &device,
        );
        let lp = model_valid.log_prob(tensor);
        let vals: Vec<f32> = lp.to_data().to_vec().unwrap();
        for &v in &vals {
            densities.push(v.exp() as f64);
        }
    }

    println!("Sampling and density evaluation done.");
    (sample_x, sample_y, densities, nx, ny, min_x, max_x, min_y, max_y)
};

Epoch    1/500  NLL: 2.1329
Epoch  100/500  NLL: 0.3120
Epoch  200/500  NLL: 0.3122
Epoch  300/500  NLL: 0.3057


## Samples vs Training Data

In [ ]:
let mut train_scatter = Curve::new();
train_scatter
    .set_line_style("None")
    .set_line_alpha(0.2)
    .set_marker_style("o")
    .set_marker_size(1.0)
    .set_marker_color("#cccccc")
    .set_marker_line_color("#cccccc")
    .set_label("training data");
train_scatter.draw(&data_x, &data_y);

let mut sample_scatter = Curve::new();
sample_scatter
    .set_line_style("None")
    .set_line_alpha(0.5)
    .set_marker_style("o")
    .set_marker_size(1.5)
    .set_marker_color("#e74c3c")
    .set_marker_line_color("#e74c3c")
    .set_label("NSF samples");
sample_scatter.draw(&sample_x, &sample_y);

let mut plot = Plot::new();
plot.add(&train_scatter)
    .add(&sample_scatter)
    .set_equal_axes(true)
    .set_labels("x", "y")
    .set_title("NSF Samples vs Training Data")
    .legend();
plot.show_in_jupyter("samples.svg")

## Learned Density (Contour Plot)

In [ ]:
use plotpy::Contour;

let mut grid_x: Vec<Vec<f64>> = Vec::new();
let mut grid_y: Vec<Vec<f64>> = Vec::new();
let mut grid_z: Vec<Vec<f64>> = Vec::new();
for iy in 0..ny {
    let yy = min_y + (max_y - min_y) * iy as f64 / (ny - 1) as f64;
    let mut row_x = Vec::with_capacity(nx);
    let mut row_y = Vec::with_capacity(nx);
    let mut row_z = Vec::with_capacity(nx);
    for ix in 0..nx {
        let xx = min_x + (max_x - min_x) * ix as f64 / (nx - 1) as f64;
        row_x.push(xx);
        row_y.push(yy);
        row_z.push(densities[iy * nx + ix]);
    }
    grid_x.push(row_x);
    grid_y.push(row_y);
    grid_z.push(row_z);
}

let mut contour = Contour::new();
contour
    .set_colormap_name("viridis")
    .set_colorbar_label("density")
    .set_no_lines(true);
contour.draw(&grid_x, &grid_y, &grid_z);

let mut pts = Curve::new();
pts.set_line_style("None")
    .set_line_alpha(0.08)
    .set_marker_style(".")
    .set_marker_size(0.5)
    .set_marker_color("white")
    .set_marker_line_color("white");
pts.draw(&data_x, &data_y);

let mut plot = Plot::new();
plot.add(&contour)
    .add(&pts)
    .set_equal_axes(true)
    .set_labels("x", "y")
    .set_title("NSF Learned Density");
plot.show_in_jupyter("density.svg")